In [ ]:
%pip install ipython-sql sqlalchemy pandas 

# Necessary Library And Data bases

In [2]:
%reload_ext sql
%sql sqlite://
%config SqlMagic.style = '_DEPRECATED_DEFAULT'


In [3]:

import pandas as pd
import sqlite3


In [ ]:
sales = pd.read_csv('data/sales.csv')



In [10]:
%sql --persist sales

 * sqlite://


'Persisted sales'

In [11]:
%%sql
select * from sales limit 5;


 * sqlite://
Done.


index,product_id,country,sales_amount
0,101,USA,500
1,102,USA,300
2,103,USA,700
3,104,USA,450
4,201,Germany,600


In [22]:
%%sql
select country, product_id, sales_amount ,rank
from
(select country, product_id, sales_amount , row_number() over (partition by country order by sales_amount desc) as rank
from sales )
where rank <= 3

 * sqlite://
Done.


country,product_id,sales_amount,rank
France,304,500,1
France,302,450,2
France,301,400,3
Germany,204,650,1
Germany,201,600,2
Germany,202,550,3
USA,103,700,1
USA,101,500,2
USA,104,450,3


In [24]:
%%sql
select * , row_number() over ( order by sales_amount desc) as rank
from sales

 * sqlite://
Done.


index,product_id,country,sales_amount,rank
2,103,USA,700,1
7,204,Germany,650,2
4,201,Germany,600,3
5,202,Germany,550,4
0,101,USA,500,5
6,203,Germany,500,6
11,304,France,500,7
3,104,USA,450,8
9,302,France,450,9
8,301,France,400,10


### What is PARTITION BY?
PARTITION BY divides the result set into groups (partitions) based on one or more columns.

The window function then operates separately within each partition, as if each group was its own small table.


### How It Works:
The OVER (PARTITION BY country ORDER BY sales_amount DESC) clause tells SQL:

Partition the table by country (USA and Germany are separate groups).

Order each group by sales_amount descending.

Assign a row number starting from 1 within each group.

In [29]:
%%sql
SELECT
  product_id,
  country,
  sales_amount,
  ROW_NUMBER() OVER (PARTITION BY country ORDER BY sales_amount DESC) AS rn
FROM sales;


 * sqlite://
Done.


product_id,country,sales_amount,rn
304,France,500,1
302,France,450,2
301,France,400,3
303,France,350,4
204,Germany,650,1
201,Germany,600,2
202,Germany,550,3
203,Germany,500,4
103,USA,700,1
101,USA,500,2


1. ROW_NUMBER()
Assigns a unique, sequential number to each row within a partition.

No ties: Even if two rows have the same value, their row numbers will be different.

2. RANK()
Gives the same rank to tied values, but the next rank is skipped accordingly (gaps in ranking).

If two rows tie for 1st, next rank will be 3.

3. DENSE_RANK()
Like RANK(), but no gaps in ranking.

If two rows tie for 1st, next rank will be 2.


You must use **ROW_NUMBER(),Rank()** with an OVER() clause.
But inside OVER(), you don’t have to use PARTITION BY.

**PARTITION** BY is not a standalone SQL clause; it only appears inside window functions (OVER()).

You can’t use PARTITION BY by itself; it must be part of something like:
ROW_NUMBER() OVER (PARTITION BY ...)

In [28]:
%%sql
SELECT
  product_id,
  sales_amount,
  ROW_NUMBER() OVER (ORDER BY sales_amount DESC) AS rn,
  RANK() OVER (ORDER BY sales_amount DESC) AS rnk,
  DENSE_RANK() OVER (ORDER BY sales_amount DESC) AS rnk2
FROM sales;

 * sqlite://
Done.


product_id,sales_amount,rn,rnk,rnk2
103,700,1,1,1
204,650,2,2,2
201,600,3,3,3
202,550,4,4,4
101,500,5,5,5
203,500,6,5,5
304,500,7,5,5
104,450,8,8,6
302,450,9,8,6
301,400,10,10,7
